# QFT Example: showcasing Julia's symbolic capabilities

> [!INFO]-
> The goal of this notebook is to illustrate the symbolic computation power of Julia which combines the easy of Python, the clarity of Mathematica and the speed of C++.

> [!NOTE]
> In this notebook we use the `Symbolics.jl` package, and we employ an example from QFT (scattering amplitudes) to illustrate its capabilities.

In **Quantum Field Theory** (QFT), one of the most common but tedious tasks is calculating scattering amplitudes (the probabilities that particles bounce off each other).

To do this, physicists use **Feynmann diagrams**, which translate visually into long mathematical traces of Dirac $\gamma$ matrices. These matrices obey a specific algebraic rule called a Clifford Algebra:

$$
\gamma^\mu \gamma ^\nu + \gamma^\nu \gamma ^\mu = 2 \eta^{\mu\nu} I
$$

Let's look at how `python` and `julia` can be used to approach this problem.

# SymPy
**SymPy** is built on `python`'s object-oriented system. To define a custom non-commutative physics algebra (where $A \times B \not = B \times A$), you have to navigate heavily nested object classes.

To simpligy an expression like $\gamma^\mu \gamma^\nu \gamma_\mu$ in SymPy, you often have to rely on the pre-built `sympy.physics.hep` sub-module. Otherwise, to custom the code, you will have to write complex custom Python classes overriding standard multiplication methods of python.

# Julia
`Symbolics.jl` uses a concept called **Term Rewriting**. Instead of building rigid object classes, simply writes down the mathematical rule exactly like on paper, and applies it to your equations.

Here is an example of the symbolic simplification of a QFT-like matrix trace in `Julia`:

In [4]:
using Symbolics

# 1. Define our symbolic spacetime indices and gamma objects
@variables μ ν
@syms γ(idx) η(idx1, idx2)

# 2. Define a mathematical rewrite rule: γ(μ)*γ(ν) + γ(ν)*γ(μ) => 2*η(μ,ν)
# In Julia, we can isolate the non-commutative swapping rule:
rule = @rule γ(~μ) * γ(~ν) => 2*η(~μ, ~ν) - γ(~ν) * γ(~μ)

# 3. Test it on an unsimplified expression: γ¹γ²
expr = γ(1) * γ(2)

# 4. Apply the rule to swap the order of the matrices
simplified = rule(expr)
println(simplified) 

2η(1, 2) - γ(1)*γ(2)


If you have different simplification identities, you can bundle them into a single object:

```julia
my_qft_rules = Rewriters.Chain([rule1, rule2, rule3])
final_amplitude = score(my_qft_rules, raw_feynman_expression)
```